In [117]:
import numpy as np

class DotsAndBoxesState:
    def __init__(self, h, w):
        self.h = h 
        self.w = w  
        
        self.horizontals = np.zeros((h, w - 1), dtype=int)
        self.verticals = np.zeros((h - 1, w), dtype=int)
        
        self.player_turn = 1
        self.scores = {1: 0, 2: 0}
        
        self.boxes = np.zeros((h - 1, w - 1), dtype=int)
    
    def apply_move(self, type, r, c):
        if type == 'h':
            self.horizontals[r][c] = 1
        else:
            self.verticals[r][c] = 1
            
        boxes_completed = self._check_and_update_boxes()
        
        # Handle turn logic
        if boxes_completed > 0:
            self.scores[self.player_turn] += boxes_completed
        else:
            self.player_turn = 3 - self.player_turn
            
        return self

    def _check_and_update_boxes(self):
        new_boxes = 0
        for r in range(self.h - 1):
            for c in range(self.w - 1):
                if self.boxes[r][c] == 0:
                    # Check 4 edges
                    top = self.horizontals[r][c]
                    bottom = self.horizontals[r+1][c]
                    left = self.verticals[r][c]
                    right = self.verticals[r][c+1]
                    
                    if top and bottom and left and right:
                        self.boxes[r][c] = self.player_turn
                        new_boxes += 1
        return new_boxes

    def get_legal_moves(self):
        moves = []
        # Horizontal edges
        for r in range(self.h):
            for c in range(self.w - 1):
                if self.horizontals[r][c] == 0:
                    moves.append(('h', r, c))
        # Vertical edges
        for r in range(self.h - 1):
            for c in range(self.w):
                if self.verticals[r][c] == 0:
                    moves.append(('v', r, c))
        return moves

    def is_terminal(self):
        # Game is over if no moves are left
        return len(self.get_legal_moves()) == 0

    def display(self):
        for r in range(self.h):
            line = ""
            for c in range(self.w - 1):
                line += "● "
                line += "— " if self.horizontals[r][c] == 1 else "  "
            line += "●"
            print(line)
            
            if r < self.h - 1:
                v_line = ""
                for c in range(self.w):
                    v_line += "| " if self.verticals[r][c] == 1 else "  "
                    if c < self.w - 1:
                        owner = self.boxes[r][c]
                        v_line += f"{owner} " if owner != 0 else "  "
                print(v_line)
    
    def clone(self):
        new_state = DotsAndBoxesState(self.h, self.w)
        new_state.horizontals = self.horizontals.copy()
        new_state.verticals = self.verticals.copy()
        new_state.boxes = self.boxes.copy()
        new_state.player_turn = self.player_turn
        new_state.scores = self.scores.copy()
        return new_state


In [118]:
# Example state and moves
state = DotsAndBoxesState(5, 5)
turn = state.player_turn

print(f"Current turn: Player {turn}")
state.apply_move('h', 0, 0)

turn = state.player_turn
print(f"Current turn: Player {turn}")
state.apply_move('v', 0, 0)

turn = state.player_turn
print(f"Current turn: Player {turn}")
state.apply_move('h', 1, 0)

turn = state.player_turn
print(f"Current turn: Player {turn}")
state.apply_move('v', 0, 1)

turn = state.player_turn
print(f"Current turn: Player {turn}")
state.display()

Current turn: Player 1
Current turn: Player 2
Current turn: Player 1
Current turn: Player 2
Current turn: Player 2
● — ●   ●   ●   ●
| 2 |             
● — ●   ●   ●   ●
                  
●   ●   ●   ●   ●
                  
●   ●   ●   ●   ●
                  
●   ●   ●   ●   ●


In [ ]:
import math
import random

class MCTSNode:
    def __init__(self, state, parent=None, move=None):
        self.state = state          
        self.parent = parent       
        self.move = move            
        self.children = []          
        
        self.visits = 0             
        self.wins = 0.0             
        
        self.untried_actions = state.get_legal_moves()

    def is_fully_expanded(self):
        return len(self.untried_actions) == 0

    def is_terminal(self):
        return self.state.is_terminal()

    def best_child(self, c_param=0.7):
        """
        UCB1 = (wins / visits) + C * sqrt(log(parent_visits) / visits)
        """
        choices_weights = [
            (child.wins / child.visits) + c_param * math.sqrt((2 * math.log(self.visits) / child.visits))
            for child in self.children
        ]
        return self.children[choices_weights.index(max(choices_weights))]

    def expand(self):
        """
        Picks an untried move, applies it to a new state, and adds a new child node to the tree.
        """
        move = self.untried_actions.pop()
        next_state = self.state.clone()
        next_state.apply_move(*move)
    
        child_node = MCTSNode(next_state, parent=self, move=move)
        self.children.append(child_node)
        
        return child_node

In [125]:
# Simulate a random playout from the given node's state

root = MCTSNode(DotsAndBoxesState(3, 3))
root.expand()
root.expand()
root.expand()

print ("After expanding root node 3 times")
print(f"Check Children: {len(root.children)}")
print (f"Untried Actions: {len(root.untried_actions)}\n")

print(f"Root state:")
root.state.display()
print("\n")

print(f"First child state:")
root.children[0].state.display()

After expanding root node 3 times
Check Children: 3
Untried Actions: 9

Root state:
●   ●   ●
          
●   ●   ●
          
●   ●   ●


First child state:
●   ●   ●
          
●   ●   ●
        | 
●   ●   ●


In [ ]:
class MCTS:
    def __init__(self, iterations=1000):
        self.iterations = iterations

    def search(self, initial_state):
        root = MCTSNode(state=initial_state)

        for _ in range(self.iterations):
            # 1. SELECTION
            node = self.select(root)
            
            # 2. EXPANSION
            if not node.is_terminal():
                node = node.expand()
            
            # 3. SIMULATION (Rollout)
            result = self.simulate(node.state)
            
            # 4. BACKPROPAGATION
            self.backpropagate(node, result)

        # After all iterations, pick the child with the most visits (not necessarily highest UCB)
        return self.best_action(root)

    def select(self, node):
        """Navigate the tree until we find a leaf or unexpanded node."""
        while node.is_fully_expanded() and not node.is_terminal():
            node = node.best_child()
        return node

    def simulate(self, state):
        temp_state = state.clone()
        while not temp_state.is_terminal():
            moves = temp_state.get_legal_moves()
            
            # Winning moves
            winning_moves = [m for m in moves if self._would_complete_box(temp_state, m)]
            if winning_moves:
                move = random.choice(winning_moves)
            else:
                # Safe moves
                safe_moves = [m for m in moves if self._is_safe_move(temp_state, m)]
                if safe_moves:
                    move = random.choice(safe_moves)
                else:
                    # Random moves
                    move = random.choice(moves)
            
            temp_state.apply_move(*move)
    
        total_boxes = (temp_state.h - 1) * (temp_state.w - 1)
    
        p1_score = temp_state.scores[1]
        p2_score = temp_state.scores[2]

        return (p1_score - p2_score) / total_boxes if total_boxes > 0 else 0

    def _count_sides(self, state, r, c):
        """Counts how many sides of box (r, c) are currently filled."""
        count = 0
        if state.horizontals[r][c] == 1: count += 1     # Top
        if state.horizontals[r+1][c] == 1: count += 1   # Bottom
        if state.verticals[r][c] == 1: count += 1       # Left
        if state.verticals[r][c+1] == 1: count += 1     # Right
        return count

    def _is_safe_move(self, state, move):
        edge_type, r, c = move
        
        # Determine which boxes this edge touches
        boxes_to_check = []
        if edge_type == 'h':
            if r > 0: boxes_to_check.append((r-1, c))    # Top
            if r < state.h - 1: boxes_to_check.append((r, c)) # Bottom
        else:
            if c > 0: boxes_to_check.append((r, c-1))    # Left
            if c < state.w - 1: boxes_to_check.append((r, c)) # Right

        for br, bc in boxes_to_check:
            if self._count_sides(state, br, bc) == 2:
                return False
        return True

    def _would_complete_box(self, state, move):
        edge_type, r, c = move
        
        if edge_type == 'h':
            # Check box ABOVE the horizontal line
            if r > 0:
                # Box is formed by: this line, line above, and two vertical sides
                if (state.horizontals[r-1][c] == 1 and 
                    state.verticals[r-1][c] == 1 and 
                    state.verticals[r-1][c+1] == 1):
                    return True
            # Check box BELOW the horizontal line
            if r < state.h - 1:
                if (state.horizontals[r+1][c] == 1 and 
                    state.verticals[r][c] == 1 and 
                    state.verticals[r][c+1] == 1):
                    return True
                    
        elif edge_type == 'v':
            # Check box to the LEFT of the vertical line
            if c > 0:
                if (state.verticals[r][c-1] == 1 and 
                    state.horizontals[r][c-1] == 1 and 
                    state.horizontals[r+1][c-1] == 1):
                    return True
            # Check box to the RIGHT of the vertical line
            if c < state.w - 1:
                if (state.verticals[r][c+1] == 1 and 
                    state.horizontals[r][c] == 1 and 
                    state.horizontals[r+1][c] == 1):
                    return True
                    
        return False

    def backpropagate(self, node, result):
        """
        Update the win/visit counts along the path from the given node back to the root.
        """
        while node is not None:
            node.visits += 1
            
            if node.parent is not None:
                moving_player = node.parent.state.player_turn
                
                # If P1 moved, they want to maximize the positive difference.
                # If P2 moved, they want to maximize the negative difference.
                if moving_player == 1:
                    node.wins += result
                else:
                    node.wins += -result
            
            node = node.parent

    def best_action(self, root):
        """Return the move that led to the most visited child."""
        best_child_node = max(root.children, key=lambda c: c.visits)
        return best_child_node.move

In [ ]:
# Setup game
game_state = DotsAndBoxesState(3, 3)

# Run MCTS
mcts_ai = MCTS(iterations=500)
best_move = mcts_ai.search(game_state)

print(f"The AI suggests moving at: {best_move}")

# Apply and visualize
game_state.apply_move(*best_move)
game_state.display()

The AI suggests moving at: ('v', 1, 2)
●   ●   ●
          
●   ●   ●
        | 
●   ●   ●


In [123]:
# Create a situation where one move completes a box
test_state = DotsAndBoxesState(3, 3)
test_state.apply_move('h', 0, 0)
test_state.apply_move('v', 0, 0)
test_state.apply_move('v', 0, 1)
# Now 'h', 1, 0 will complete the box.

mcts = MCTS(iterations=1000)
best_move = mcts.search(test_state)

print(f"AI chose: {best_move}")
assert best_move == ('h', 1, 0), "AI failed to take the obvious box!"

AI chose: ('h', 1, 0)


In [ ]:
import time
from IPython.display import clear_output

def play_ai_vs_ai(size=5, iterations=2000):
    state = DotsAndBoxesState(size, size)
    mcts = MCTS(iterations=iterations)
    
    print("Starting AI vs AI Match...")
    time.sleep(1)

    try:
        while not state.is_terminal():
            clear_output(wait=True)
            
            print(f"--- AI vs AI: {size}x{size} Dots ---")
            print(f"Player 1 Score: {state.scores[1]}")
            print(f"Player 2 Score: {state.scores[2]}")
            print(f"Current Turn: Player {state.player_turn}")
            print("-" * 25)
            state.display()
            print("-" * 25)

            start_time = time.time()
            best_move = mcts.search(state)
            end_time = time.time()

            print(f"AI Player {state.player_turn} chose {best_move} (computed in {end_time - start_time:.2f}s)")    
            state.apply_move(*best_move)
            time.sleep(1)

        clear_output(wait=True)
        print("--- GAME OVER ---")
        state.display()
        print("-" * 25)
        print(f"FINAL SCORE - Player 1: {state.scores[1]} | Player 2: {state.scores[2]}")
        
        if state.scores[1] > state.scores[2]:
            print("WINNER: PLAYER 1")
        elif state.scores[2] > state.scores[1]:
            print("WINNER: PLAYER 2")
        else:
            print("RESULT: IT'S A DRAW!")

    except KeyboardInterrupt:
        print("\nSimulation stopped by user.")

play_ai_vs_ai(size=5, iterations=2000)

--- AI vs AI: 5x5 Dots ---
Player 1 Score: 0
Player 2 Score: 0
Current Turn: Player 1
-------------------------
●   ●   ●   ●   ●
                  
●   ●   ●   ●   ●
                  
●   ●   ●   ●   ●
                  
●   ●   ●   ●   ●
                  
●   ●   ●   ●   ●
-------------------------

Simulation stopped by user.
